In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loder = PyPDFLoader('telecom_guide.pdf')
pages = loder.load()


print(f"loaded {len(pages)} pages from PDF .")
print(" \n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

C:\Users\sidda\AppData\Local\Temp\ipykernel_33000\3391004304.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


loaded 9 pages from PDF .
 
--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [3]:
print(pages[1].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600,
    chunk_overlap = 100,
    separators=["\n\n", "\n",".", " " ],
)

chunks = splitter.split_documents(pages)

len(chunks)

37

In [5]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page': 1, 'page_label': '2'}, page_content='Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G

In [6]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [7]:
# Embeding and storing them to chromaDB


from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [9]:
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
vector_store = Chroma.from_documents(chunks , embeddings)


print(f"Vector store ready. {vector_store._collection.count()} vectors stored")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready. 74 vectors stored


In [12]:
# Retriever 

retriever = vector_store.as_retriever(search_kwargs = {"k":3})

test_query = 'What is VoLTE and how dies it imrove call  quality?'
retrived = retriever.invoke(test_query)

for i , doc in enumerate(retrived , 1):
    print(f'--- Chunk {i} ---')
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 3 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt



In [13]:
# System Prompt

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


# Helper : join retrived chunks into a single context string 
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs )




SYSTEM_PROMPT = """ \
You are a helpful telecom assistant 
Answer the question using ONLY the comntext provided below . 
if the context does not contain enough information , say so clearly 

context : {context}
"""


prompt = ChatPromptTemplate.from_messages([
    ('system',SYSTEM_PROMPT),
    ('human',"{questions}"),
]) 



# llm via Groq API 
llm = ChatGroq(model='qwen/qwen3-32b',
               temperature=0,
               reasoning_format='parsed'
               )

chain = (
    {'context':retriever | format_docs, 'questions':RunnablePassthrough()}
    | prompt
    | llm
    |StrOutputParser()
)



print(' RAG chain Assembled')


 RAG chain Assembled


In [15]:
question = ' How does international roaming work and what changes should I expect ?'

print(f' Q :{question}\n')

print(" A :" ,chain.invoke(question))

 Q : How does international roaming work and what changes should I expect ?

 A : International roaming allows your device to connect to a partner network in a foreign country when you're outside your home network's coverage. Here's how it works and what to expect:

### **How It Works**  
1. **Authentication**: The visited network verifies your subscription using signaling protocols (SS7 or Diameter) to communicate with your home network.  
2. **Authorization**: Your home network validates your account and grants access to services (voice, data, SMS).  

### **Changes to Expect**  
- **Activation Requirement**: Roaming must be enabled **before departure** via the MyTelecom app (Plan & Services > International Roaming) or by dialing 611. Activation may take up to **15 minutes**.  
- **Post-Arrival Activation**: If enabled after landing, you may experience a **brief service interruption** while your HLR (Home Location Register) record updates.  
- **Roaming Bundles**:  
  - Bundles are t